<a href="https://colab.research.google.com/github/sumitsingh63/EAA_Project/blob/main/01_data_integration_and_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Environment initialized successfully.")

Environment initialized successfully.


In [2]:
# GitHub repository information

GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "sumitsingh63/EAA_Project/main/data/raw/"
)

IMD_FILE = GITHUB_RAW_BASE + "imd_rainfall_master_1901_2025.csv"
NAKSHATRA_FILE = GITHUB_RAW_BASE + "nakshatra_tags.csv"

print("GitHub data paths configured.")
print("IMD file:", IMD_FILE)
print("Nakshatra file:", NAKSHATRA_FILE)

GitHub data paths configured.
IMD file: https://raw.githubusercontent.com/sumitsingh63/EAA_Project/main/data/raw/imd_rainfall_master_1901_2025.csv
Nakshatra file: https://raw.githubusercontent.com/sumitsingh63/EAA_Project/main/data/raw/nakshatra_tags.csv


In [3]:
# Load datasets from GitHub

imd_df = pd.read_csv(IMD_FILE)
nakshatra_df = pd.read_csv(NAKSHATRA_FILE)

print("Datasets loaded successfully.")
print()
print("IMD shape:", imd_df.shape)
print("Nakshatra shape:", nakshatra_df.shape)

Datasets loaded successfully.

IMD shape: (45656, 5)
Nakshatra shape: (45656, 7)


Inspect the IMD Dataset

In [4]:
print("IMD DATASET")
print("=" * 50)

print("Columns:")
print(imd_df.columns.tolist())

print("\nFirst 5 rows:")
display(imd_df.head())

print("\nData types:")
print(imd_df.dtypes)

IMD DATASET
Columns:
['Date', 'Year', 'Month', 'Day', 'Rainfall_mm']

First 5 rows:


,Date,Year,Month,Day,Rainfall_mm
0,1901-01-01,1901,1,1,5.38
1,1901-01-02,1901,1,2,1.85
2,1901-01-03,1901,1,3,1.05
3,1901-01-04,1901,1,4,1.79
4,1901-01-05,1901,1,5,1.16



Data types:
Date            object
Year             int64
Month            int64
Day              int64
Rainfall_mm    float64
dtype: object


Inspect the Nakshatra Dataset

In [5]:
print("NAKSHATRA DATASET")
print("=" * 50)

print("Columns:")
print(nakshatra_df.columns.tolist())

print("\nFirst 5 rows:")
display(nakshatra_df.head())

print("\nData types:")
print(nakshatra_df.dtypes)

NAKSHATRA DATASET
Columns:
['date', 'sun_sidereal_longitude', 'solar_nakshatra', 'solar_nakshatra_pada', 'moon_sidereal_longitude', 'lunar_nakshatra', 'lunar_nakshatra_pada']

First 5 rows:


,date,sun_sidereal_longitude,solar_nakshatra,solar_nakshatra_pada,moon_sidereal_longitude,lunar_nakshatra,lunar_nakshatra_pada
0,1901-01-01,257.4304,Purva Ashadha,2,25.8374,Bharani,4
1,1901-01-02,258.4496,Purva Ashadha,2,39.9232,Krittika,4
2,1901-01-03,259.4686,Purva Ashadha,2,53.9119,Mrigashira,1
3,1901-01-04,260.4877,Purva Ashadha,3,67.7524,Ardra,1
4,1901-01-05,261.5066,Purva Ashadha,3,81.3910,Punarvasu,1



Data types:
date                        object
sun_sidereal_longitude     float64
solar_nakshatra             object
solar_nakshatra_pada         int64
moon_sidereal_longitude    float64
lunar_nakshatra             object
lunar_nakshatra_pada         int64
dtype: object


Check Date Ranges

In [6]:
print("IMD date range:")
print(imd_df["Date"].min(), "to", imd_df["Date"].max())

print("\nNakshatra date range:")
print(nakshatra_df["date"].min(), "to", nakshatra_df["date"].max())

IMD date range:
1901-01-01 to 2025-12-31

Nakshatra date range:
1901-01-01 to 2025-12-31


Convert date columns to pandas datetime



In [7]:
# Convert date columns to pandas datetime

imd_df["Date"] = pd.to_datetime(imd_df["Date"])
nakshatra_df["date"] = pd.to_datetime(nakshatra_df["date"])

print("Date conversion completed.")
print("IMD:", imd_df["Date"].dtype)
print("Nakshatra:", nakshatra_df["date"].dtype)

Date conversion completed.
IMD: datetime64[ns]
Nakshatra: datetime64[ns]


Check Duplicate Dates

In [8]:
# Check whether each dataset contains exactly one record per day

imd_duplicates = imd_df["Date"].duplicated().sum()
nakshatra_duplicates = nakshatra_df["date"].duplicated().sum()

print("IMD duplicate dates:", imd_duplicates)
print("Nakshatra duplicate dates:", nakshatra_duplicates)

IMD duplicate dates: 0
Nakshatra duplicate dates: 0


Check whether the data sets are identical

In [9]:
# Compare the actual dates in both datasets

imd_dates = set(imd_df["Date"])
nakshatra_dates = set(nakshatra_df["date"])

imd_missing_from_nakshatra = imd_dates - nakshatra_dates
nakshatra_missing_from_imd = nakshatra_dates - imd_dates

print("IMD dates missing from Nakshatra dataset:",
      len(imd_missing_from_nakshatra))

print("Nakshatra dates missing from IMD dataset:",
      len(nakshatra_missing_from_imd))

IMD dates missing from Nakshatra dataset: 0
Nakshatra dates missing from IMD dataset: 0


In [10]:
# Merge IMD rainfall data with astronomical/Nakshatra data

master_df = pd.merge(
    imd_df,
    nakshatra_df,
    left_on="Date",
    right_on="date",
    how="inner",
    validate="one_to_one"
)

# Remove the duplicate date column
master_df = master_df.drop(columns=["date"])

print("Datasets merged successfully.")
print("Master dataset shape:", master_df.shape)

Datasets merged successfully.
Master dataset shape: (45656, 11)


In [11]:
print("MASTER DATASET")
print("=" * 60)

print("Shape:", master_df.shape)

print("\nColumns:")
for i, column in enumerate(master_df.columns, start=1):
    print(f"{i}. {column}")

print("\nFirst 5 records:")
display(master_df.head())

MASTER DATASET
Shape: (45656, 11)

Columns:
1. Date
2. Year
3. Month
4. Day
5. Rainfall_mm
6. sun_sidereal_longitude
7. solar_nakshatra
8. solar_nakshatra_pada
9. moon_sidereal_longitude
10. lunar_nakshatra
11. lunar_nakshatra_pada

First 5 records:


,Date,Year,Month,Day,Rainfall_mm,sun_sidereal_longitude,solar_nakshatra,solar_nakshatra_pada,moon_sidereal_longitude,lunar_nakshatra,lunar_nakshatra_pada
0,1901-01-01,1901,1,1,5.38,257.4304,Purva Ashadha,2,25.8374,Bharani,4
1,1901-01-02,1901,1,2,1.85,258.4496,Purva Ashadha,2,39.9232,Krittika,4
2,1901-01-03,1901,1,3,1.05,259.4686,Purva Ashadha,2,53.9119,Mrigashira,1
3,1901-01-04,1901,1,4,1.79,260.4877,Purva Ashadha,3,67.7524,Ardra,1
4,1901-01-05,1901,1,5,1.16,261.5066,Purva Ashadha,3,81.3910,Punarvasu,1


In [12]:
print("DATA QUALITY CHECK")
print("=" * 60)

print("\nMissing values:")
print(master_df.isnull().sum())

print("\nDuplicate dates:")
print(master_df["Date"].duplicated().sum())

print("\nDate range:")
print(
    master_df["Date"].min(),
    "to",
    master_df["Date"].max()
)

DATA QUALITY CHECK

Missing values:
Date                       0
Year                       0
Month                      0
Day                        0
Rainfall_mm                0
sun_sidereal_longitude     0
solar_nakshatra            0
solar_nakshatra_pada       0
moon_sidereal_longitude    0
lunar_nakshatra            0
lunar_nakshatra_pada       0
dtype: int64

Duplicate dates:
0

Date range:
1901-01-01 00:00:00 to 2025-12-31 00:00:00


In [13]:
print("NUMERICAL SUMMARY")
print("=" * 60)

display(master_df.describe())

NUMERICAL SUMMARY


,Date,Year,Month,Day,Rainfall_mm,sun_sidereal_longitude,solar_nakshatra_pada,moon_sidereal_longitude,lunar_nakshatra_pada
count,45656,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000
mean,1963-07-02 12:00:00,1963.000679,6.522954,15.729565,3.129317,178.115809,2.500504,179.993029,2.501402
min,1901-01-01 00:00:00,1901.000000,1.000000,1.000000,0.000000,0.014400,1.000000,0.002500,1.000000
25%,1932-04-01 18:00:00,1932.000000,4.000000,8.000000,0.450000,87.824100,2.000000,90.086600,2.000000
50%,1963-07-02 12:00:00,1963.000000,7.000000,16.000000,1.600000,176.217050,3.000000,179.987300,3.000000
75%,1994-10-01 06:00:00,1994.000000,10.000000,23.000000,5.100000,268.423700,4.000000,269.929750,4.000000
max,2025-12-31 00:00:00,2025.000000,12.000000,31.000000,19.130000,359.996200,4.000000,359.983200,4.000000
std,NaN,36.083435,3.448734,8.800158,3.483631,104.106729,1.118027,103.899121,1.118418


Basic Numerical Summary

In [14]:
print("NUMERICAL SUMMARY")
print("=" * 60)

display(master_df.describe())

NUMERICAL SUMMARY


,Date,Year,Month,Day,Rainfall_mm,sun_sidereal_longitude,solar_nakshatra_pada,moon_sidereal_longitude,lunar_nakshatra_pada
count,45656,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000,45656.000000
mean,1963-07-02 12:00:00,1963.000679,6.522954,15.729565,3.129317,178.115809,2.500504,179.993029,2.501402
min,1901-01-01 00:00:00,1901.000000,1.000000,1.000000,0.000000,0.014400,1.000000,0.002500,1.000000
25%,1932-04-01 18:00:00,1932.000000,4.000000,8.000000,0.450000,87.824100,2.000000,90.086600,2.000000
50%,1963-07-02 12:00:00,1963.000000,7.000000,16.000000,1.600000,176.217050,3.000000,179.987300,3.000000
75%,1994-10-01 06:00:00,1994.000000,10.000000,23.000000,5.100000,268.423700,4.000000,269.929750,4.000000
max,2025-12-31 00:00:00,2025.000000,12.000000,31.000000,19.130000,359.996200,4.000000,359.983200,4.000000
std,NaN,36.083435,3.448734,8.800158,3.483631,104.106729,1.118027,103.899121,1.118418


In [15]:
from pathlib import Path

processed_dir = Path("/content/data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

master_file = processed_dir / "master_dataset_1901_2025.csv"

master_df.to_csv(master_file, index=False)

print("Master dataset saved successfully.")
print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))
print("File:", master_file)

Master dataset saved successfully.
Rows: 45656
Columns: 11
File: /content/data/processed/master_dataset_1901_2025.csv
